# Assignment 3 Template: 2 FNNs

mandatory installs!

In [0]:
!pip install tensorflow
%restart_python

## 0) Load Delta table

In [0]:
TABLE = "workspace.default.cb_1_cb_2_features_v_2"
SAMPLE_FRAC = 0.15
RANDOM_SEED = 42

# Use this experiment path in your own Databricks workspace
MLFLOW_EXPERIMENT = "/Users/kole.guenther@uhsp.edu/Assignment3"

df_spark = spark.table(TABLE)
cols = [
    "molecule_chembl_id",
    "cb1_p", "cb2_p", "delta_p_cb1_minus_cb2",
    "cb1_active", "cb2_active", "selectivity_direction",
    "morgan_fp_str", "maccs_fp_str", "rdkit_desc50_str"
]

pdf = (
    df_spark.select(*[c for c in cols if c in df_spark.columns])
    .sample(withReplacement=False, fraction=SAMPLE_FRAC, seed=RANDOM_SEED)
    .toPandas()
)

print("Rows in pandas sample:", len(pdf))
pdf.head()

## 1) Parse JSON arrays and build descriptor matrices

In [0]:
import json
import random
import numpy as np
import pandas as pd


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    try:
        import tensorflow as tf

        tf.random.set_seed(seed)
    except Exception:
        pass


def parse_json_array(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return None
    try:
        return json.loads(x)
    except Exception:
        return None


pdf["morgan_fp"] = pdf["morgan_fp_str"].apply(parse_json_array)
pdf["maccs_fp"] = pdf["maccs_fp_str"].apply(parse_json_array)
pdf["rdkit_desc50"] = pdf["rdkit_desc50_str"].apply(parse_json_array)

pdf = pdf.dropna(subset=["morgan_fp", "maccs_fp", "rdkit_desc50"]).reset_index(drop=True)

X_morgan = np.vstack(pdf["morgan_fp"].apply(lambda v: np.array(v, dtype=np.float32)))
X_maccs = np.vstack(pdf["maccs_fp"].apply(lambda v: np.array(v, dtype=np.float32)))
X_desc = np.vstack(pdf["rdkit_desc50"].apply(lambda v: np.array(v, dtype=np.float32)))
X_all = np.hstack([X_desc, X_maccs, X_morgan])

FEATURES = {
    "desc": X_desc,
    "maccs": X_maccs,
    "morgan": X_morgan,
    "all": X_all,
}

for name, mat in FEATURES.items():
    print(f"{name:>6}: {mat.shape}")

## 2) FNN utility functions (shared by regression + classification)

In [0]:
import mlflow
import mlflow.tensorflow
from sklearn.model_selection import KFold, ParameterGrid
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
)
from sklearn.preprocessing import StandardScaler, LabelEncoder

import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping


mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment(MLFLOW_EXPERIMENT)


def make_fnn_regressor(input_dim, hidden_layers=(256, 128), activation="relu", dropout=0.2, learning_rate=1e-3):
    model = Sequential()
    model.add(Dense(hidden_layers[0], activation=activation, input_shape=(input_dim,)))
    model.add(BatchNormalization())
    model.add(Dropout(dropout))
    for units in hidden_layers[1:]:
        model.add(Dense(units, activation=activation))
        model.add(BatchNormalization())
        model.add(Dropout(dropout))
    model.add(Dense(1, activation="linear"))
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss="mse",
        metrics=[tf.keras.metrics.RootMeanSquaredError(name="rmse"), "mae"],
    )
    return model


def make_fnn_classifier(
    input_dim,
    n_classes,
    hidden_layers=(256, 128),
    activation="relu",
    dropout=0.2,
    learning_rate=1e-3,
):
    model = Sequential()
    model.add(Dense(hidden_layers[0], activation=activation, input_shape=(input_dim,)))
    model.add(BatchNormalization())
    model.add(Dropout(dropout))
    for units in hidden_layers[1:]:
        model.add(Dense(units, activation=activation))
        model.add(BatchNormalization())
        model.add(Dropout(dropout))

    if n_classes == 2:
        model.add(Dense(1, activation="sigmoid"))
        loss = "binary_crossentropy"
    else:
        model.add(Dense(n_classes, activation="softmax"))
        loss = "sparse_categorical_crossentropy"

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss=loss,
        metrics=["accuracy"],
    )
    return model

## 3) Regression template cell: CV + grid search + feature comparisons

Set `REG_TARGET` to one of:
- `cb1_p`
- `cb2_p`
- `delta_p_cb1_minus_cb2`

In [0]:
import matplotlib.pyplot as plt
REG_TARGET = "cb1_p"

reg_grid = {
    "hidden_layers": [(64,), (128, 64)],
    "activation": ["relu", "tanh"],
    "dropout": [0.2],
    "learning_rate": [1e-3, 5e-4],
    "batch_size": [32],
    "epochs": [20]
}

set_seed(RANDOM_SEED)
y_reg = pd.to_numeric(pdf[REG_TARGET], errors="coerce")
mask_reg = y_reg.notna()
y_reg = y_reg.loc[mask_reg].to_numpy(dtype=np.float32)

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

reg_results = []
reg_fold_results = []

for feature_name, X_full in FEATURES.items():
    X_reg = X_full[mask_reg.to_numpy()]

    scaler = StandardScaler()
    X_reg = scaler.fit_transform(X_reg)

    best_cv_rmse = np.inf
    best_config = None
    best_cv_metrics = None

    for params in ParameterGrid(reg_grid):
        fold_rmses = []
        fold_maes = []
        fold_r2s = []

        for train_idx, valid_idx in cv.split(X_reg, y_reg):
            X_tr, X_va = X_reg[train_idx], X_reg[valid_idx]
            y_tr, y_va = y_reg[train_idx], y_reg[valid_idx]

            model = make_fnn_regressor(
                input_dim=X_reg.shape[1],
                hidden_layers=params["hidden_layers"],
                activation=params["activation"],
                dropout=params["dropout"],
                learning_rate=params["learning_rate"],
            )

            es = EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True)
            model.fit(
                X_tr,
                y_tr,
                validation_data=(X_va, y_va),
                epochs=params["epochs"],
                batch_size=params["batch_size"],
                verbose=0,
                callbacks=[es],
            )

            y_va_pred = model.predict(X_va, verbose=0).reshape(-1)
            rmse = float(np.sqrt(mean_squared_error(y_va, y_va_pred)))
            mae = float(mean_absolute_error(y_va, y_va_pred))
            r2 = float(r2_score(y_va, y_va_pred))

            fold_rmses.append(rmse)
            fold_maes.append(mae)
            fold_r2s.append(r2)

            reg_fold_results.append({
                "feature_set": feature_name,
                "params": str(params),
                "fold": len(fold_rmses),
                "rmse": rmse,
                "mae": mae,
                "r2": r2
            })

        cv_rmse_mean = np.mean(fold_rmses)
        cv_rmse_std = np.std(fold_rmses)

        cv_mae_mean = np.mean(fold_maes)
        cv_mae_std = np.std(fold_maes)

        cv_r2_mean = np.mean(fold_r2s)
        cv_r2_std = np.std(fold_r2s)
        if cv_rmse_mean < best_cv_rmse:
            best_cv_rmse = cv_rmse_mean
            best_config = params
            best_cv_metrics = {
                "cv_rmse_mean": cv_rmse_mean,
                "cv_rmse_std": cv_rmse_std,
                "cv_mae_mean": cv_mae_mean,
                "cv_mae_std": cv_mae_std,
                "cv_r2_mean": cv_r2_mean,
                "cv_r2_std": cv_r2_std,
            }

    # Final fit and metrics metrics
    final_model = make_fnn_regressor(
        input_dim=X_reg.shape[1],
        hidden_layers=best_config["hidden_layers"],
        activation=best_config["activation"],
        dropout=best_config["dropout"],
        learning_rate=best_config["learning_rate"],
    )
    history = final_model.fit(
        X_reg,
        y_reg,
        epochs=best_config["epochs"],
        batch_size=best_config["batch_size"],
        validation_split=0.2,
        verbose=0,
    )

    y_hat = final_model.predict(X_reg, verbose=0).reshape(-1)

    residuals = y_reg - y_hat

    # Predicted vs Actual Plot
    plt.figure(figsize=(6, 5))
    plt.scatter(y_reg, y_hat, alpha=0.6)
    plt.xlabel("Actual CB1 Binding Affinity")
    plt.ylabel("Predicted CB1 Binding Affinity")
    plt.title(f"Predicted vs Actual - {feature_name}")

    min_val = min(y_reg.min(), y_hat.min())
    max_val = max(y_reg.max(), y_hat.max())
    plt.plot([min_val, max_val], [min_val, max_val], linestyle="--")

    plt.show()

    # Residual Plot
    plt.figure(figsize=(6, 5))
    plt.scatter(y_hat, residuals, alpha=0.6)
    plt.axhline(0, linestyle="--")
    plt.xlabel("Actual Binding Affinity (CB1)")
    plt.ylabel("Predicted Binding Affinity (CB1)")
    plt.title(f"Residual Plot - {feature_name}")
    plt.show()

    # Training / Validation Curve
    plt.figure(figsize=(6, 5))
    plt.plot(history.history["loss"], label="Training Loss")
    plt.plot(history.history["val_loss"], label="Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("MSE Loss")
    plt.title(f"Training vs Validation Loss - {feature_name}")
    plt.legend()
    plt.show()

    rmse = float(np.sqrt(mean_squared_error(y_reg, y_hat)))
    mae = float(mean_absolute_error(y_reg, y_hat))
    r2 = float(r2_score(y_reg, y_hat))

    with mlflow.start_run(run_name=f"FNN_REG_{REG_TARGET}_{feature_name}"):
        mlflow.log_param("task", "regression")
        mlflow.log_param("target", REG_TARGET)
        mlflow.log_param("feature_set", feature_name)
        mlflow.log_param("cv_folds", 5)
        for k, v in best_config.items():
            mlflow.log_param(f"best_{k}", str(v))

        for metric_name, metric_value in best_cv_metrics.items():
            mlflow.log_metric(metric_name, float(metric_value))
        mlflow.log_metric("train_rmse", rmse)
        mlflow.log_metric("train_mae", mae)
        mlflow.log_metric("train_r2", r2)

    reg_results.append(
        {
            "feature_set": feature_name,
            "target": REG_TARGET,
            **best_cv_metrics,
            "rmse_full": rmse,
            "mae_full": mae,
            "r2_full": r2,
            "best_config": best_config,
        }
    )

reg_fold_df = pd.DataFrame(reg_fold_results)
display(reg_fold_df)

pd.DataFrame(reg_results).sort_values("cv_rmse_mean")

## 4) Classification template cell: CV + grid search + feature comparisons

Set `CLS_TARGET` to one of:
- `cb1_active` (binary)
- `cb2_active` (binary)
- `selectivity_direction` (multi-class)

In [0]:
from sklearn.metrics import confusion_matrix, roc_curve
import matplotlib.pyplot as plt

CLS_TARGET = "cb1_active"
ACTIVE_THRESHOLD = 6.0

pdf[CLS_TARGET] = np.where(
    pd.to_numeric(pdf["cb1_p"], errors="coerce") >= ACTIVE_THRESHOLD,
    1,
    0
)

cls_grid = {
    "hidden_layers": [(64,), (128, 64)],
    "activation": ["relu", "tanh"],
    "dropout": [0.2],
    "learning_rate": [1e-3, 5e-4],
    "batch_size": [32],
    "epochs": [20]
}

set_seed(RANDOM_SEED)

if CLS_TARGET == "cb1_active":
    y_cls_raw = pd.to_numeric(pdf[CLS_TARGET], errors="coerce")
    mask_cls = y_cls_raw.notna()
    y_cls = y_cls_raw.loc[mask_cls].astype(int).to_numpy()
    n_classes = 2
else:
    y_cls_raw = pdf[CLS_TARGET].astype(str).str.strip()
    y_cls_raw = y_cls_raw.replace({"": np.nan, "None": np.nan, "nan": np.nan, "NaN": np.nan})
    mask_cls = y_cls_raw.notna()
    y_clean = y_cls_raw.loc[mask_cls]
    encoder = LabelEncoder()
    y_cls = encoder.fit_transform(y_clean)
    n_classes = len(np.unique(y_cls))

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

cls_results = []
cls_fold_results = []

for feature_name, X_full in FEATURES.items():
    X_cls = X_full[mask_cls.to_numpy()]

    scaler = StandardScaler()
    X_cls = scaler.fit_transform(X_cls)

    best_cv_f1 = -np.inf
    best_config = None
    best_cv_metrics = None

    for params in ParameterGrid(cls_grid):
        fold_accs = []
        fold_precs = []
        fold_recs = []
        fold_f1s = []
        fold_aucs = []

        for train_idx, valid_idx in cv.split(X_cls, y_cls):
            X_tr, X_va = X_cls[train_idx], X_cls[valid_idx]
            y_tr, y_va = y_cls[train_idx], y_cls[valid_idx]

            model = make_fnn_classifier(
                input_dim=X_cls.shape[1],
                n_classes=n_classes,
                hidden_layers=params["hidden_layers"],
                activation=params["activation"],
                dropout=params["dropout"],
                learning_rate=params["learning_rate"],
            )

            es = EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True)
            model.fit(
                X_tr,
                y_tr,
                validation_data=(X_va, y_va),
                epochs=params["epochs"],
                batch_size=params["batch_size"],
                verbose=0,
                callbacks=[es],
            )

            if n_classes == 2:
                y_va_prob = model.predict(X_va, verbose=0).reshape(-1)
                y_va_pred = (y_va_prob >= 0.5).astype(int)
                avg_type = "binary"
                auc = float(roc_auc_score(y_va, y_va_prob))
            else:
                y_va_prob = model.predict(X_va, verbose=0)
                y_va_pred = np.argmax(y_va_prob, axis=1)
                avg_type = "macro"
                auc = np.nan

            acc = float(accuracy_score(y_va, y_va_pred))
            prec = float(precision_score(y_va, y_va_pred, average=avg_type, zero_division=0))
            rec = float(recall_score(y_va, y_va_pred, average=avg_type, zero_division=0))
            f1 = float(f1_score(y_va, y_va_pred, average=avg_type, zero_division=0))

            fold_accs.append(acc)
            fold_precs.append(prec)
            fold_recs.append(rec)
            fold_f1s.append(f1)

            if not np.isnan(auc):
                fold_aucs.append(auc)

            cls_fold_results.append({
                "feature_set": feature_name,
                "params": str(params),
                "fold": len(fold_f1s),
                "accuracy": acc,
                "precision": prec,
                "recall": rec,
                "f1": f1,
                "roc_auc": auc,
            })

        cv_accuracy_mean = float(np.mean(fold_accs))
        cv_accuracy_std = float(np.std(fold_accs))

        cv_precision_mean = float(np.mean(fold_precs))
        cv_precision_std = float(np.std(fold_precs))

        cv_recall_mean = float(np.mean(fold_recs))
        cv_recall_std = float(np.std(fold_recs))

        cv_f1_mean = float(np.mean(fold_f1s))
        cv_f1_std = float(np.std(fold_f1s))

        cv_auc_mean = float(np.mean(fold_aucs)) if len(fold_aucs) > 0 else np.nan
        cv_auc_std = float(np.std(fold_aucs)) if len(fold_aucs) > 0 else np.nan

        if cv_f1_mean > best_cv_f1:
            best_cv_f1 = cv_f1_mean
            best_config = params
            best_cv_metrics = {
                "cv_accuracy_mean": cv_accuracy_mean,
                "cv_accuracy_std": cv_accuracy_std,
                "cv_precision_mean": cv_precision_mean,
                "cv_precision_std": cv_precision_std,
                "cv_recall_mean": cv_recall_mean,
                "cv_recall_std": cv_recall_std,
                "cv_f1_mean": cv_f1_mean,
                "cv_f1_std": cv_f1_std,
                "cv_roc_auc_mean": cv_auc_mean,
                "cv_roc_auc_std": cv_auc_std,
            }

    final_model = make_fnn_classifier(
        input_dim=X_cls.shape[1],
        n_classes=n_classes,
        hidden_layers=best_config["hidden_layers"],
        activation=best_config["activation"],
        dropout=best_config["dropout"],
        learning_rate=best_config["learning_rate"],
    )
    history = final_model.fit(
        X_cls,
        y_cls,
        epochs=best_config["epochs"],
        batch_size=best_config["batch_size"],
        validation_split=0.2,
        verbose=0,
    )

    if n_classes == 2:
        y_prob = final_model.predict(X_cls, verbose=0).reshape(-1)
        y_pred = (y_prob >= 0.5).astype(int)
        auc = float(roc_auc_score(y_cls, y_prob))
        f1_avg = "binary"
    else:
        y_prob = final_model.predict(X_cls, verbose=0)
        y_pred = np.argmax(y_prob, axis=1)
        auc = np.nan
        f1_avg = "macro"

    acc = float(accuracy_score(y_cls, y_pred))
    prec = float(precision_score(y_cls, y_pred, average=f1_avg, zero_division=0))
    rec = float(recall_score(y_cls, y_pred, average=f1_avg, zero_division=0))
    f1 = float(f1_score(y_cls, y_pred, average=f1_avg, zero_division=0))
    mcc = float(matthews_corrcoef(y_cls, y_pred))

    # Confusion Matrix
    cm = confusion_matrix(y_cls, y_pred)

    plt.figure(figsize=(5, 4))
    plt.imshow(cm)
    plt.title(f"Confusion Matrix - {feature_name}")
    plt.xlabel("Predicted Label")
    plt.ylabel("Actual Label")
    plt.colorbar()

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, cm[i, j], ha="center", va="center")

    plt.show()

    # ROC Curve for Binary Classification
    if n_classes == 2:
        fpr, tpr, thresholds = roc_curve(y_cls, y_prob)

        plt.figure(figsize=(6, 5))
        plt.plot(fpr, tpr, label=f"ROC-AUC = {auc:.3f}")
        plt.plot([0, 1], [0, 1], linestyle="--")
        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate")
        plt.title(f"ROC Curve - {feature_name}")
        plt.legend()
        plt.show()

    # Training / Validation Curve
    plt.figure(figsize=(6, 5))
    plt.plot(history.history["loss"], label="Training Loss")
    plt.plot(history.history["val_loss"], label="Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"Training vs Validation Loss - {feature_name}")
    plt.legend()
    plt.show()

    with mlflow.start_run(run_name=f"FNN_CLS_{CLS_TARGET}_{feature_name}"):
        mlflow.log_param("task", "classification")
        mlflow.log_param("target", CLS_TARGET)
        mlflow.log_param("feature_set", feature_name)
        mlflow.log_param("n_classes", n_classes)
        mlflow.log_param("cv_folds", 5)
        for k, v in best_config.items():
            mlflow.log_param(f"best_{k}", str(v))

        for metric_name, metric_value in best_cv_metrics.items():
            if not np.isnan(metric_value):
                mlflow.log_metric(metric_name, float(metric_value))
        mlflow.log_metric("accuracy_full", acc)
        mlflow.log_metric("precision_full", prec)
        mlflow.log_metric("recall_full", rec)
        mlflow.log_metric("f1_full", f1)
        mlflow.log_metric("mcc_full", mcc)
        if not np.isnan(auc):
            mlflow.log_metric("roc_auc_full", auc)

        cls_results.append(
            {
                "feature_set": feature_name,
                "target": CLS_TARGET,
                **best_cv_metrics,
                "train_accuracy": acc,
                "train_precision": prec,
                "train_recall": rec,
                "train_f1": f1,
                "train_mcc": mcc,
                "train_roc_auc": auc,
                "best_config": best_config,
            }
        )

cls_fold_df = pd.DataFrame(cls_fold_results)
display(cls_fold_df)

pd.DataFrame(cls_results).sort_values("cv_f1_mean", ascending=False)

## 5) Notes for assignment write-up

- Report the **best feature set** by CV metric (RMSE for regression, F1 for classification).
- Include the **best hyperparameters** from grid search.
- Compare at least two targets (e.g., `cb1_p` and `selectivity_direction`).
- Include MLflow screenshots/tables with run IDs, parameters, and metrics.